In [1]:
import sys
BASE_DIR = "../../../.."
sys.path.insert(0, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import json
import torch
from time import time
import os
import gc
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass

random.seed(42)

from src.agents.hosted import CustomAgent
from src.utils import ReaderMetrics
from src.utils.inference_metrics import compute_predictive_entropy

CONTEXTS_DATASET_PATH = "../../../../data/mtssquad/contexts.csv"
QA_DATASET_PATH = "../../../../data/mtssquad/qa_dataset.csv"
AGENT_MODEL_PATH = "../../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf"

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
!pip install sentence_transformers
!pip install chromadb
!pip install Levenshtein
!pip install langchain_huggingface
!pip install torchmetrics
!pip install evaluate
!pip install accelerate>=0.26.0
!pip install nltk

In [2]:
# RU PROMPTS
# "Ты — AI-помощник, который помогает решать возникающие проблемы."
# 'Ответь на вопрос, используя доступную информацию из текстов в списке ниже. Если в списке нет текстов, достаточно релевантных для генерации ответа на их основе, то сгенерируй следующий текст: "У меня нет ответа на ваш вопрос". Сгенерируй ответ на русском языке. Не дублируй вопрос в ответе. Сгенерируй только ответ на указанный вопрос. Ответ должен быть кратким. Не генерируй ничего лишнего.'
# "У меня нет ответа на ваш вопрос"
# "{user_p}\n\nДоступная информация:\n{cnt_list}\n\nВопрос:\n{q}\nОтвет:\n"
# EN PROMPTS
# "You are an AI assistant who helps solve user issues."
# 'Answer the question using the available information from the texts in the list below. If there are no texts in the list that are relevant enough to generate answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer in Russian. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.'
# "I do not have an answer to your question"
# "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n"

In [3]:
PARAMS = {
    'version': "1",
    'num_samples': 150,
    'num_contexts': 1,
    'model': AGENT_MODEL_PATH,
    'system_prompt': "Ты — AI-помощник, который помогает решать возникающие проблемы.",
    "item_format": "- {document}",
    "user_prompt": 'Ответь на вопрос, используя доступную информацию из текстов в списке ниже. Если в списке нет текстов, достаточно релевантных для генерации ответа на их основе, то сгенерируй следующий текст: "У меня нет ответа на ваш вопрос". Сгенерируй ответ на русском языке. Не дублируй вопрос в ответе. Сгенерируй только ответ на указанный вопрос. Ответ должен быть кратким. Не генерируй ничего лишнего.',
    "prompt_format": "{user_p}\n\nДоступная информация:\n{cnt_list}\n\nВопрос:\n{q}\nОтвет:\n",
    'scores': {'rel': 1.0, 'unrel': 0.0},
    'gen_strat': {'max_new_tokens': 1024, 'do_sample': False, 'num_beams': 1},
    'stub_answer': "У меня нет ответа на ваш вопрос",
    'calculate_entropy': True
}

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'
LOGS_SAVE_DIR = './logs_v2'
META_INFO_DIR_NAME = 'gen_metainfo'

if os.path.exists(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}'):
    print("Dir exists")
else:
    print("Creating Dir...")
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}')
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}/{META_INFO_DIR_NAME}')

Dir exists


### Подключение к агенту

In [4]:
agent = CustomAgent(PARAMS['model'], output_logits=PARAMS['calculate_entropy'], use_cache=True, output_attentions=False, output_scores=False, output_hidden_states=False)
output = agent.generate(user_prompt="what is wrong with humanity?", system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
print(output[0])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

What a profound and complex question! As a neutral AI assistant, I'll provide a balanced and non-judgmental perspective. Humanity is a diverse and multifaceted species, and it's challenging to pinpoint a single "wrong" aspect. However, I can highlight some common issues and challenges that humanity faces:

1. **Conflict and violence**: Wars, terrorism, and interpersonal violence are ongoing problems that cause immense suffering and loss of life.
2. **Environmental degradation**: Human activities have led to climate change, pollution, deforestation, and species extinction, threatening the planet's ecological balance.
3. **Inequality and social injustice**: Systemic inequalities based on race, gender, class, religion, and other factors perpetuate discrimination, poverty, and marginalization.
4. **Mental health and well-being**: Many people struggle with mental health issues, such as depression, anxiety, and trauma, which can have a significant impact on their lives and relationships.
5. 

### Формируем список контекстов для каждого запроса со скорами

In [5]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [6]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    cur_list_ids = [(-1, dataset_df['relevant_context_id'][i])]
    CONTEXTS_LIST_IDS.append(cur_list_ids)

100%|██████████| 150/150 [00:00<00:00, 25655.33it/s]


In [7]:
print(CONTEXTS_LIST_IDS[0])

[(-1, 0)]


### Готовим промпт

In [8]:
contexts_df = pd.read_csv(CONTEXTS_DATASET_PATH)

In [9]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    rel_doc = contexts_df['context'][CONTEXTS_LIST_IDS[i][0][1]]
    documents_list = PARAMS['item_format'].format(document=rel_doc)
    
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=dataset_df['question'][i]))

100%|██████████| 150/150 [00:00<00:00, 102034.64it/s]


In [10]:
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [11]:
print(USER_PROMPTS[0])

Ответь на вопрос, используя доступную информацию из текстов в списке ниже. Если в списке нет текстов, достаточно релевантных для генерации ответа на их основе, то сгенерируй следующий текст: "У меня нет ответа на ваш вопрос". Сгенерируй ответ на русском языке. Не дублируй вопрос в ответе. Сгенерируй только ответ на указанный вопрос. Ответ должен быть кратким. Не генерируй ничего лишнего.

Доступная информация:
-  Everybody , как и хотела Мадонна, выпускают синглом. При нулевом бюджете на раскрутку фото певицы решают не помещать на обложке, чтобы не отпугнуть цветную аудиторию якобы негритянской диско-соул-певицы . Everybody поднимается на 3-е место в чарте Hot Dance Club Songs, а потом на 107 место в основном, немного не дотянув до первой сотни Hot 100 журнала Billboard[91]. Менеджмент считает это отличным результатом, учитывая нулевые затраты на пиар, и хочет убедиться, что взлёт Everybody не случаен. По просьбе Мадонны вместо Каминса берут более опытного штатного аранжировщика Warner

In [12]:
del contexts_df
gc.collect()

66

### Генерируем ответы на вопросы

In [13]:
generate_answers, calc_metrics = [], []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer, meta_info = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])

    cur_metrics = dict()
    if PARAMS['calculate_entropy']:
        logits = torch.cat(meta_info['logits'], 0).cpu().detach()
        entropy = compute_predictive_entropy(logits)
        cur_metrics['predictive_entropy'] = float(entropy)
    calc_metrics.append(cur_metrics)
    generate_answers.append(pred_answer)
    
    # logits = torch.cat(meta_info['logits'], 0).cpu().detach().numpy()
    # logits_int8 = logits.astype('int8') 
    # token_logits = {f"token_{i}": token_logits for i, token_logits in enumerate(logits_int8)}
    # pa_table = pa.table(token_logits)
    # pa.parquet.write_table(pa_table, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/logits_{i}.parquet")
    
    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['answer'][i]}\nMETRICS: {cur_metrics}")
e_time = time()

  1%|▏         | 2/150 [00:01<01:17,  1.90it/s]


[0]: 
GEN: Да, по просьбе Мадонны вместо Каминса был взят более опытный аранжировщик Регги Лукас.
GOLD: Да, по просьбе Мадонны заменили аранжировщика Каминса на более опытного штатного аранжировщика Warner Bros. Records Регги Лукаса.
METRICS: {'predictive_entropy': 6.541418552398682}


 67%|██████▋   | 101/150 [00:45<00:35,  1.36it/s]


[100]: 
GEN: Деятельность официальных органов по получению и применению финансовых средств осуществляется для выполнения надлежащих функций государства.
GOLD: Деятельность официальных органов по получению и применению финансовых средств осуществляется для выполнения надлежащих функций.
METRICS: {'predictive_entropy': 4.919223785400391}


100%|██████████| 150/150 [01:07<00:00,  2.23it/s]


In [14]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    formated_contexts = [(float(item[0]), int(item[1])) for item in CONTEXTS_LIST_IDS[i]]
    cur_item = {
        'gen_answer': str(generate_answers[i]), 
        'metainfo': calc_metrics[i], 
        'used_contexts': formated_contexts}
    gen_info.append(cur_item)

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [15]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
LOADING_VERSION = "1"

In [17]:
with open(f'{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [18]:
metrics = ReaderMetrics(base_dir=BASE_DIR, model_path='ru_electra_medium')

Loading Meteor...
Loading ExactMatch


In [19]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [20]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 10

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])


    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)

    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 150/150 [00:03<00:00, 38.95it/s, BLEU2=0.00236, BLEU1=0.00369, ExactMatch=0, METEOR=0.00749, BertScore=nan, Levenshtain=37.9, ROUGEL=0]


In [21]:
with open(f"{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))